In [1]:
!pip uninstall -y transformers accelerate datasets evaluate tokenizers sentencepiece peft
!pip uninstall -y bitsandbytes transformers peft accelerate

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: datasets 4.8.3
Uninstalling datasets-4.8.3:
  Successfully uninstalled datasets-4.8.3
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: sentencepiece 0.2.1
Uninstalling sentencepiece-0.2.1:
  Successfully uninstalled sentencepiece-0.2.1
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1


In [2]:
!pip install -q transformers==4.51.3
!pip install -q peft==0.15.2
!pip install -q accelerate==1.6.0
!pip install -q bitsandbytes==0.45.5

!pip install -q \
datasets==2.18.0 \
evaluate==0.4.1 \
rouge_score==0.1.2 \
sentencepiece==0.2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 88.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
torchtune 0.6.1 requires sentencepiece, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/

In [3]:
# ===============================
# IMPORT LIBRARIES
# ===============================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import nltk
from datasets import load_dataset, Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    pipeline,
    set_seed
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# Initialize NLTK
nltk.download("punkt")

import torch
torch.cuda.empty_cache()

2026-05-09 17:58:31.305824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778349511.503567      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778349511.561531      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778349512.034649      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778349512.034686      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778349512.034689      23 computation_placer.cc:177] computation placer alr

In [4]:
!kaggle datasets download -d gowrishankarp/newspaper-text-summarization-cnn-dailymail

Dataset URL: https://www.kaggle.com/datasets/gowrishankarp/newspaper-text-summarization-cnn-dailymail
License(s): CC0-1.0
100%|█████████████████████████████████████████| 503M/503M [00:02<00:00, 196MB/s]



In [5]:
!unzip newspaper-text-summarization-cnn-dailymail.zip

Archive:  newspaper-text-summarization-cnn-dailymail.zip
  inflating: cnn_dailymail/test.csv  
  inflating: cnn_dailymail/train.csv  
  inflating: cnn_dailymail/validation.csv  


In [6]:
train_df = pd.read_csv("/kaggle/working/cnn_dailymail/train.csv")
valid_df = pd.read_csv("/kaggle/working/cnn_dailymail/validation.csv")
test_df = pd.read_csv("/kaggle/working/cnn_dailymail/test.csv")


In [7]:
def clean_html(text):
    text = re.sub(r"<.*?>", "", text)  # remove HTML tags
    text = re.sub(r"&amp;", "&", text)
    text = re.sub(r"&lt;", "<", text)
    text = re.sub(r"&gt;", ">", text)
    return text


def is_valid(example):
    return len(example["article"]) > 50 and len(example["highlights"]) > 10

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9.,!?\'\" ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_repetition(text):
    text = re.sub(r"(\.){2,}", ".", text)
    text = re.sub(r"(!){2,}", "!", text)
    text = re.sub(r"\?{2,}", "?", text)
    return text

def clean_summary(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def preprocess(example):

    article = example["article"]
    summary = example["highlights"]

    article = clean_html(article)
    article = clean_repetition(article)
    article = normalize_text(article)

    summary = clean_html(summary)
    summary = clean_repetition(summary)
    summary = normalize_text(summary)

    return {
        "article": article,
        "highlights": summary
    }

In [8]:
train_dataset = train_df.sample(n=4000, random_state=42).reset_index(drop=True)


test_dataset = test_df.sample(n=100, random_state=42).reset_index(drop=True)


train_dataset[["article", "highlights"]] = train_dataset.apply(
    preprocess,
    axis=1,
    result_type="expand"
)

# ===============================
# CHECK RESULT
# ===============================

print(train_dataset.head())

                                         id  \
0  ed0fed726929c1eeabe6c390e47128dbb7d7a055   
1  023cd84001b33aed4ff0f3f5ecb0fdd2151cf543   
2  6a70a0d8d3ed365fe1df6d35f1587a8b9b298618   
3  b37204c13ea38b511265e41ac69fb12acfb63f85   
4  c24e5805afd5145bc48410e876db91d44a06be5e   

                                             article  \
0  by . mia de graaf . britons flocked to beaches...   
1  a couple who weighed a combined 32st were sham...   
2  video footage shows the heart stopping moment ...   
3  istanbul, turkey cnn about 250 people raced ac...   
4  by . daily mail reporter . published . 12 53 e...   

                                          highlights  
0  people enjoyed temperatures of 17c at brighton...  
1  couple started piling on pounds after the birt...  
2  a 17 year old boy suffering lacerations to his...  
3  syrians citizens hightail it to turkey . most ...  
4  the xue long had provided the helicopter that ...  


In [9]:
from datasets import Dataset

# ===============================
# CONVERT PANDAS -> HF DATASET
# ===============================

train_hf_dataset = Dataset.from_pandas(train_dataset)

print(train_dataset.head())

                                         id  \
0  ed0fed726929c1eeabe6c390e47128dbb7d7a055   
1  023cd84001b33aed4ff0f3f5ecb0fdd2151cf543   
2  6a70a0d8d3ed365fe1df6d35f1587a8b9b298618   
3  b37204c13ea38b511265e41ac69fb12acfb63f85   
4  c24e5805afd5145bc48410e876db91d44a06be5e   

                                             article  \
0  by . mia de graaf . britons flocked to beaches...   
1  a couple who weighed a combined 32st were sham...   
2  video footage shows the heart stopping moment ...   
3  istanbul, turkey cnn about 250 people raced ac...   
4  by . daily mail reporter . published . 12 53 e...   

                                          highlights  
0  people enjoyed temperatures of 17c at brighton...  
1  couple started piling on pounds after the birt...  
2  a 17 year old boy suffering lacerations to his...  
3  syrians citizens hightail it to turkey . most ...  
4  the xue long had provided the helicopter that ...  


In [10]:
max_input_length = 256
max_target_length = 64

def preprocess_function(examples, tokenizer):

    inputs = examples["article"]
    targets = examples["highlights"]

    # Tokenize input articles
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    # Tokenize summaries
    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [11]:
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

tokenized_dataset = train_hf_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=train_hf_dataset.column_names
)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [12]:
lora_project_path = "/kaggle/working/lora_summarizer_model"
os.makedirs(lora_project_path, exist_ok=True)

In [13]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)


In [14]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 407,470,080 || trainable%: 0.2895


In [15]:

training_args = TrainingArguments(
    output_dir=lora_project_path,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    num_train_epochs=2,

    logging_steps=50,
    save_steps=500,

    fp16=True,

    report_to="none"
)



In [16]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [17]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

/tmp/ipykernel_23/507548412.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSeq2SeqLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [18]:
print(tokenized_dataset[0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [19]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,3.332000
100,2.657200
150,2.192800
200,2.023300
250,1.955300
300,1.937400
350,1.887400
400,1.863500
450,1.878600
500,1.896000


TrainOutput(global_step=500, training_loss=2.1623651733398437, metrics={'train_runtime': 1741.383, 'train_samples_per_second': 4.594, 'train_steps_per_second': 0.287, 'total_flos': 4348704718848000.0, 'train_loss': 2.1623651733398437, 'epoch': 2.0})

In [20]:
trainer.model.save_pretrained(lora_project_path)
tokenizer.save_pretrained(lora_project_path)

('/kaggle/working/lora_summarizer_model/tokenizer_config.json',
 '/kaggle/working/lora_summarizer_model/special_tokens_map.json',
 '/kaggle/working/lora_summarizer_model/vocab.json',
 '/kaggle/working/lora_summarizer_model/merges.txt',
 '/kaggle/working/lora_summarizer_model/added_tokens.json',
 '/kaggle/working/lora_summarizer_model/tokenizer.json')

In [21]:
def summarize_article(article):

    inputs = tokenizer(
        article,
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        min_length=40,
        num_beams=4
    )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [22]:
article = """
The government announced a new economic reform plan on Monday aimed at boosting
national productivity and stabilizing inflation. According to officials,
the reform package includes tax adjustments, infrastructure investment,
and support for small businesses. Economists say the plan could stimulate
growth but warn that its success depends on implementation and global
economic conditions. Critics argue that the reforms may not adequately
address unemployment and rising living costs faced by many households.
"""

inputs = tokenizer(
    article,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
)

inputs = inputs.to(device)
print(inputs)

{'input_ids': tensor([[    0, 50118,   133,   168,   585,    10,    92,   776,  3114,   563,
            15,   302,  3448,    23, 11606, 50118, 11535,  8106,     8, 12964,
          2787,  2680,     4,   767,     7,   503,     6, 50118,   627,  3114,
          3737,  1171,   629, 11431,     6,  2112,   915,     6, 50118,   463,
           323,    13,   650,  1252,     4, 17833,  1952,   224,     5,   563,
           115, 19770, 50118, 14596,    53, 11345,    14,    63,  1282,  7971,
            15,  5574,     8,   720, 50118, 12063,  1274,     4, 11943,  5848,
            14,     5,  4907,   189,    45, 17327, 50118, 44547,  5755,     8,
          2227,  1207,  1042,  2713,    30,   171,  8195,     4, 50118,     2]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [23]:
summary_ids = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=150,
    min_length=40,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)

In [24]:
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

the reform package includes tax adjustments, infrastructure investment, and support for small businesses. Critics argue that the reforms may not adequatelyaddress unemployment and rising living costs faced by many households. Economists say the plan could stimulate


In [25]:
!zip -r /kaggle/working/lora_summarizer_model.zip /kaggle/working/lora_summarizer_model

  adding: kaggle/working/lora_summarizer_model/ (stored 0%)
  adding: kaggle/working/lora_summarizer_model/tokenizer.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 82%)
  adding: kaggle/working/lora_summarizer_model/adapter_config.json (deflated 55%)
  adding: kaggle/working/lora_summarizer_model/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/lora_summarizer_model/README.md (deflated 66%)
  adding: kaggle/working/lora_summarizer_model/merges.txt (deflated 53%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/ (stored 0%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/tokenizer.json (deflated 82%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/scheduler.pt (deflated 62%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/optimizer.pt (deflated 8%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/adapter_config.json (deflated 55%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/lora_summarizer_model/checkpoint-500/README.md (deflated 66%)
  adding: kaggle/working/lora_summariz

In [26]:
from IPython.display import FileLink

FileLink("/kaggle/working/lora_summarizer_model.zip")

/kaggle/working/lora_summarizer_model.zip